# A2 — Knowledge-Base Demo (fill this)
Show OCR quality on a sample and one working retrieval example.

# Knowledge Base Demo
Part 1: OCR quality on a sample of pages.
Part 2: one working retrieval example against the persisted FAISS index.

Assumes `build_knowledge_base(cfg)` has already been run and
`data/interim/index/{index.faiss,chunks.json}` exist.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from PIL import Image

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))  # match pytest's pythonpath = ["src"]

CORPUS_DIR = REPO_ROOT / "data" / "raw" / "bk2"

with open(REPO_ROOT / "configs/config.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print(f"index_dir: {cfg['paths']['index_dir']}")
print(f"embed model: {cfg['embed']['model']} (dim={cfg['embed']['dim']})")

## Part 1 — OCR quality on a sample

In [ ]:
def resolve_tesseract() -> tuple[str, str]:
    env_prefix = Path(
        os.environ.get("TESSERACT_ENV_PREFIX", Path.home() / ".local" / "share" / "tesseract-env")
    )
    candidates = [env_prefix / "bin" / "tesseract", env_prefix / "Library" / "bin" / "tesseract.exe"]
    tessdata_candidates = [env_prefix / "share" / "tessdata", env_prefix / "Library" / "share" / "tessdata"]
    for c in candidates:
        if c.exists():
            return str(c), str(next((t for t in tessdata_candidates if t.exists()), ""))
    which = shutil.which("tesseract")
    if which:
        return which, ""
    raise RuntimeError("tesseract not found")


TESSERACT_CMD, TESSDATA_DIR = resolve_tesseract()
LANG = cfg["ocr"]["lang"]


def ocr_text(path: Path) -> str:
    cmd = [TESSERACT_CMD, str(path), "stdout", "-l", LANG]
    if TESSDATA_DIR:
        cmd.extend(["--tessdata-dir", TESSDATA_DIR])
    result = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
    return result.stdout.strip()


SAMPLE_SIZE = 5
start_idx = 35
sample_pages = sorted(CORPUS_DIR.glob("*.png"))[start_idx:start_idx+SAMPLE_SIZE]

for page in sample_pages:
    with Image.open(page) as img:
        plt.figure(figsize=(6, 8))
        plt.imshow(img.convert("L"), cmap="gray")
        plt.title(page.name)
        plt.axis("off")
        plt.show()
    text = ocr_text(page)
    print(f"--- OCR output: {page.name} ({len(text)} chars) ---")
    print(text[:800] + ("..." if len(text) > 800 else ""))
    print()

## Part 2 — retrieval demo against the persisted index

In [ ]:
from doc_agent.index import store  # adjust import path if the package name differs

index, chunks = store.load(cfg)
print(f"Loaded index: {index.ntotal} vectors, dim={index.d}")
print(f"Loaded {len(chunks)} chunks (metadata)")
chunks[0]  # inspect the shape of a Chunk

In [ ]:
from doc_agent.retrieval.retriever import Retriever, is_weak, top_score

retriever = Retriever(cfg)

In [ ]:
def show_results(query: str, k: int = 5) -> None:
    results = retriever.retrieve(query, k=k)
    print(f'Query: "{query}"')
    print(f"top_score={top_score(results):.3f}  weak={is_weak(results, cfg)}  "
          f"(weak_threshold={cfg['retrieve']['weak_threshold']})\n")
    for rank, chunk in enumerate(results, start=1):
        preview = chunk.text.strip().replace("\n", " ")
        preview = preview[:200] + ("..." if len(preview) > 200 else "")
        print(f"[{rank}] score={chunk.score:.3f}  id={chunk.id}  pages={chunk.page_ids}")
        print(f"    {preview}\n")
    return results

In [ ]:
# Working retrieval example — swap in a query relevant to corpus
demo_results = show_results("veratrum album dosage for cholera")

In [ ]:
# Second example, deliberately checking the evidence-gating utilities the agent uses
weak_query_results = show_results("something unrelated to homeopathy remedies at all")